In [ ]:
pip install ndfu

**Imports**

In [ ]:
import numpy as np
import pandas as pd
import itertools
import random
import matplotlib.pyplot as plt
import os
from ndfu import dfu

In [ ]:
# Define the dimensions
dimensions = {
    "gender": ["male", "female", "non-binary"],
    "politics": ["left", "center", "right"],
    "age": ["<25", "25-50", ">50"],
    "education": ["low", "medium", "high"],
    "orientation": ["heterosexual", "lgbtq+"]
}

In [ ]:
class AnnotatorPool:

    def __init__(self, dimensions, exclude=None, annotators_per_identity=10,
                 scale=5, toxic_range=(4, 5), civil_range=(1, 2), neutral_range=(3, 3)):
        # Demographic configuration
        self.annotators_per_identity = annotators_per_identity
        self.identities, self.active_dims = self._get_identities(dimensions, exclude)
        self.pool = self._build_pool()

        # Rating scale configuration
        self.scale = scale
        self.toxic_range = toxic_range
        self.civil_range = civil_range
        self.neutral_range = neutral_range

    def _get_identities(self, dimensions, exclude=None):
        # Filter out excluded dimensions and generate all identity combinations
        active_dims = {k: v for k, v in dimensions.items() if k not in (exclude or [])}
        identities = [dict(zip(active_dims.keys(), combo)) for combo in itertools.product(*active_dims.values())]
        return identities, active_dims

    def _build_pool(self):
        # Assign a unique ID to each annotator and replicate each identity
        pool = []
        for identity in self.identities:
            for _ in range(self.annotators_per_identity):
                pool.append(identity.copy())
        pool = pd.DataFrame(pool)
        pool.index.name = "annotator_id"
        return pool

    def _generate_bias_config(self, polarizing_prob=0.7):
      # Randomly assign a role to each active dimension for a dataset instance
      config = {}
      for dim, values in self.active_dims.items():
          role = random.choices(["polarizing", "unimodal"], weights=[polarizing_prob, 1 - polarizing_prob])[0]
          if role == "polarizing":
              # Randomly shuffle values and split into toxic/civil poles
              shuffled = values.copy()
              random.shuffle(shuffled)
              split = random.randint(1, len(shuffled) - 1)
              config[dim] = {
                  "role": "polarizing",
                  "toxic_pole": shuffled[:split],
                  "civil_pole": shuffled[split:]
              }
          else:
              config[dim] = {
                  "role": "unimodal",
                  "convergence": random.choice(["toxic", "civil", "neutral"])
              }
      return config

    def _annotate(self, annotator, bias_config, noise=0.1):
      # Accumulate pole votes from polarizing dimensions only
      votes = []
      for dim, config in bias_config.items():
          if config["role"] == "polarizing":
              if annotator[dim] in config["toxic_pole"]:
                  votes.append("toxic")
              elif annotator[dim] in config["civil_pole"]:
                  votes.append("civil")

      # If no polarizing dimensions, fall back to unimodal convergence
      if not votes:
          for dim, config in bias_config.items():
              if config["role"] == "unimodal":
                  votes.append(config["convergence"])

      # Majority vote determines the rating range
      toxic_votes = votes.count("toxic")
      civil_votes = votes.count("civil")
      neutral_votes = votes.count("neutral")

      if toxic_votes > civil_votes and toxic_votes > neutral_votes:
          rating_range = self.toxic_range
      elif civil_votes > toxic_votes and civil_votes > neutral_votes:
          rating_range = self.civil_range
      else:
          rating_range = self.neutral_range

      # Inject noise: with probability noise, sample from full scale instead
      if random.random() < noise:
          return random.randint(1, self.scale)

      return random.randint(*rating_range)


    def generate_dataset(self, n_texts=100, n_annotators_per_text=100, noise=0.1, polarizing_prob=0.7):
      # Generate a random bias config for this dataset instance
      bias_config = self._generate_bias_config(polarizing_prob)

      records = []
      for text_id in range(n_texts):
          # Sample a subset of annotators from the pool for this text
          sampled = self.pool.sample(n=n_annotators_per_text, replace=False)
          for annotator_id, annotator in sampled.iterrows():
              rating = self._annotate(annotator, bias_config, noise)
              records.append({
                  "text_id": text_id,
                  "annotator_id": annotator_id,
                  **annotator.to_dict(),
                  "rating": rating
              })

      return pd.DataFrame(records), bias_config

In [ ]:
pool = AnnotatorPool(dimensions)
dataset, bias_config = pool.generate_dataset(n_texts=10, n_annotators_per_text=100)

print(dataset.head(10))
print("\nBias config:")
for dim, config in bias_config.items():
    print(f"  {dim}: {config}")

   text_id  annotator_id      gender politics    age education   orientation  \
0        0           946      female    right    <25      high  heterosexual   
1        0          1463  non-binary    right    <25    medium  heterosexual   
2        0           383        male    right    <25    medium  heterosexual   
3        0           382        male    right    <25    medium  heterosexual   
4        0           353        male   center    >50      high        lgbtq+   
5        0           745      female   center    <25    medium  heterosexual   
6        0           257        male   center  25-50       low        lgbtq+   
7        0          1111  non-binary     left    <25    medium        lgbtq+   
8        0           971      female    right  25-50       low        lgbtq+   
9        0           129        male     left    >50       low  heterosexual   

   rating  
0       5  
1       4  
2       4  
3       5  
4       4  
5       1  
6       2  
7       5  
8       4  

In [ ]:
def compute_ndfu(ratings, scale=5):
    # Convert ratings to a normalized histogram and compute nDFU
    counts = np.bincount(ratings, minlength=scale + 1)[1:]  # ignore 0
    hist = counts / counts.sum()
    return dfu(hist)

def analyze_dataset(dataset, bias_config, active_dims, scale=5):
    results = {}

    for text_id, text_data in dataset.groupby("text_id"):
        text_results = {}

        # Overall nDFU for this text
        text_results["overall"] = compute_ndfu(text_data["rating"].values, scale)

        # nDFU per dimension value
        for dim in active_dims:
            text_results[dim] = {}
            for value, group in text_data.groupby(dim):
                text_results[dim][value] = compute_ndfu(group["rating"].values, scale)

        results[text_id] = text_results

    return results

results = analyze_dataset(dataset, bias_config, dimensions)

# Print summary for first text
text_0 = results[0]
print(f"Overall nDFU: {text_0['overall']:.3f}")
for dim, values in text_0.items():
    if dim == "overall":
        continue
    role = bias_config[dim]["role"]
    print(f"\n{dim} ({role}):")
    for value, ndfu in values.items():
        print(f"  {value}: {ndfu:.3f}")

Overall nDFU: 0.262

gender (unimodal):
  female: 0.455
  male: 0.176
  non-binary: 0.214

politics (polarizing):
  center: 0.667
  left: 0.176
  right: 0.118

age (polarizing):
  25-50: 0.500
  <25: 0.250
  >50: 0.143

education (unimodal):
  high: 0.357
  low: 0.357
  medium: 0.286

orientation (polarizing):
  heterosexual: 0.556
  lgbtq+: 0.042
